In [ ]:
# Anchor-only baseline: predicts last_known_TVT_input for every hidden row.
# Used to verify Kaggle scoring path is functional.

import os, glob
from pathlib import Path
import numpy as np
import pandas as pd

# Locate competition data
INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = None
for root, dirs, _files in os.walk(INPUT_ROOT):
    depth = root.count("/") - str(INPUT_ROOT).count("/")
    if depth > 3:
        dirs[:] = []
        continue
    if "test" in dirs and "train" in dirs:
        DATA_ROOT = root
        break
if DATA_ROOT is None:
    raise SystemExit("FATAL: no test/+train/ found")

TEST_DIR = Path(DATA_ROOT) / "test"
print(f"DATA_ROOT={DATA_ROOT}")
print(f"test wells: {len(list(TEST_DIR.glob('*__horizontal_well.csv')))}")

rows = []
for h_path in sorted(TEST_DIR.glob("*__horizontal_well.csv")):
    wid = h_path.name.replace("__horizontal_well.csv", "")
    df = pd.read_csv(h_path)
    if "TVT_input" not in df.columns:
        continue
    tvt_in = df["TVT_input"].to_numpy(dtype=np.float64)
    finite = np.isfinite(tvt_in)
    if finite.any():
        last = float(tvt_in[np.flatnonzero(finite)[-1]])
    else:
        last = 11354.51  # train-median fallback
    eval_idx = np.flatnonzero(~finite)
    for i in eval_idx:
        rows.append({"id": f"{wid}_{int(i)}", "tvt": last})

sub = pd.DataFrame(rows)
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"submission rows: {len(sub)}")
print(sub.head())
